# Práctica 8 — Deconvolución inmune: citometría digital con TumorDecon

**Módulo de Transcriptómica — Máster en Bioinformática**

Una muestra de RNA-seq bulk de tumor es una **mezcla** de células tumorales y de muchos tipos celulares del microambiente (inmunitarios, estroma…). La **deconvolución** estima la proporción de cada tipo celular a partir de la expresión bulk, usando una **matriz de signaturas** de referencia.

En esta práctica usamos el paquete **TumorDecon** (Le et al., 2020, *Briefings in Bioinformatics*) sobre datos de **TCGA COADREAD** (cáncer colorrectal, 592 pacientes) y la matriz de signaturas **LM22** (22 tipos de células inmunes de CIBERSORT).

**Métodos que compararemos:**

| Método | Tipo | Entrada | Salida |
|---|---|---|---|
| **CIBERSORT** | Regresión (ν-SVR) | Matriz de signaturas | Fracciones (suman 1) |
| **DeconRNASeq** | Regresión (mínimos cuadrados no negativos) | Matriz de signaturas | Fracciones |
| **ssGSEA** | Basado en rangos | Gene sets up-regulados | Scores de enriquecimiento |
| **SingScore** | Basado en rangos | Gene sets up-regulados | Scores de enriquecimiento |

> **Pregunta central**: ¿dan los cuatro métodos resultados coherentes sobre la infiltración inmune del tumor colorrectal?

## 1. Instalación

TumorDecon requiere la librería `singscore` (fork de kristyhoran) que no está en PyPI, así que la instalamos desde GitHub. Tarda 1–2 minutos.

In [ ]:
!pip install -q git+https://github.com/kristyhoran/singscore
!pip install -q TumorDecon
!pip install -q "pandas==2.2.3"

import importlib.metadata
print("TumorDecon", importlib.metadata.version("TumorDecon"))

### Parche de compatibilidad

TumorDecon es un paquete de 2020 que usa dos APIs eliminadas en las versiones actuales de Python/pandas:

1. `collections.Mapping` → ahora vive en `collections.abc` (lo necesita el método ssGSEA)
2. `DataFrame.append()` → eliminado en pandas 2.x (lo necesita singscore)

Aplicamos dos *shims* mínimos para que funcione (no modifican los resultados, solo restauran los nombres antiguos):

In [ ]:
# --- Parche de compatibilidad para Python 3.10+ y pandas 2.x ---
import collections
import collections.abc

for _name in ["Mapping", "MutableMapping", "Sequence", "Iterable", "Callable"]:
    if not hasattr(collections, _name):
        setattr(collections, _name, getattr(collections.abc, _name))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if not hasattr(pd.DataFrame, "append"):
    def _df_append(self, other, **kwargs):
        return pd.concat([self, other])
    pd.DataFrame.append = _df_append

import TumorDecon as td
print("TumorDecon listo")

## 2. Datos

- **Mezcla (mixture)**: expresión RNA-seq de 592 tumores de cáncer colorrectal de TCGA (COADREAD, vía cBioPortal).
  > **Nota**: la matriz original (17.493 genes, 100 MB) se ha reducido a **5.411 genes** (los 483 genes de LM22 presentes + los 5.000 más variables) para que sea manejable en GitHub/Colab. Esto **no afecta a CIBERSORT ni DeconRNASeq** (solo usan los genes de la signatura; lo comprobamos: resultados idénticos), pero sí aproxima los métodos de rangos (ssGSEA/SingScore), que rankearían sobre todo el transcriptoma.
- **LM22**: matriz de signaturas de 22 tipos de células inmunes (547 genes) de CIBERSORT.
- **`kmeans_signature_matrix_qval.txt`**: matriz de signaturas alternativa (16 subtipos celulares, derivada de datos single-cell) para la sección opcional.

In [ ]:
import urllib.request
import gzip
import shutil
from pathlib import Path

DATA_DIR = Path("datos_deconvolucion")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/victorsanchezarevalo/transcriptomica-master/main/practicas/08-deconvolution-tumordecon/datos"
FICHEROS = ["coadread_tcga_reducida.txt.gz", "LM22.txt", "kmeans_signature_matrix_qval.txt"]

for f in FICHEROS:
    destino = DATA_DIR / f
    if not destino.exists():
        print(f"Descargando {f} ...")
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", destino)
    print(f"{f}: {destino.stat().st_size / 1e6:.2f} MB")

# Descomprimir la matriz de expresión
gz = DATA_DIR / "coadread_tcga_reducida.txt.gz"
txt = DATA_DIR / "coadread_tcga_reducida.txt"
if not txt.exists():
    print("Descomprimiendo ...")
    with gzip.open(gz, "rb") as f_in, open(txt, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
print("\nDatos listos.")

In [ ]:
# Cargar la mezcla y la matriz de signaturas
rna = td.read_rna_file(DATA_DIR / "coadread_tcga_reducida.txt")
print(f"Mezcla: {rna.shape[0]} genes x {rna.shape[1]} pacientes")

sig = td.read_sig_file(DATA_DIR / "LM22.txt")
print(f"LM22: {sig.shape[0]} genes x {sig.shape[1]} tipos celulares")

comunes = sig.index.intersection(rna.index)
print(f"\nGenes de LM22 presentes en la mezcla: {len(comunes)} de {sig.shape[0]}")
print("\nTipos celulares de LM22:")
print(list(sig.columns))

## 3. Modelos lineales: CIBERSORT y DeconRNASeq

Ambos modelan la mezcla como **combinación lineal** de las signaturas y resuelven las proporciones por regresión (CIBERSORT usa ν-SVR; DeconRNASeq, mínimos cuadrados no negativos). Devuelven **fracciones** que suman ~1 por paciente.

> CIBERSORT sobre los 592 pacientes tarda ~2–3 minutos en Colab.

In [ ]:
# CIBERSORT sobre TODOS los pacientes
ciber_freqs = td.tumor_deconvolve(
    rna, "cibersort",
    patient_IDs="ALL",
    sig_matrix=sig,
    args={"nu": "best", "scaling": "minmax"},
)
print("CIBERSORT:", ciber_freqs.shape)
ciber_freqs.to_csv("cibersort_results.csv", index_label="Patient_ID")
ciber_freqs.iloc[:5, :8]

In [ ]:
# DeconRNASeq sobre TODOS los pacientes
decon_freqs = td.tumor_deconvolve(
    rna, "DeconRNASeq",
    patient_IDs="ALL",
    sig_matrix=sig,
    args={"scaling": "minmax", "print_progress": False},
)
print("DeconRNASeq:", decon_freqs.shape)
decon_freqs.to_csv("deconrnaseq_results.csv", index_label="Patient_ID")
decon_freqs.iloc[:5, :8]

### ¿Coinciden los dos métodos?

Comparamos las fracciones estimadas por CIBERSORT y DeconRNASeq para cada tipo celular:

In [ ]:
# Correlación entre métodos por tipo celular
import scipy.stats as st

corrs = {}
for ct in ciber_freqs.columns:
    r, _ = st.pearsonr(ciber_freqs[ct], decon_freqs[ct])
    corrs[ct] = r
corrs = pd.Series(corrs).sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
colors = ["firebrick" if r < 0.5 else "steelblue" for r in corrs]
ax.barh(corrs.index, corrs.values, color=colors, edgecolor="k")
ax.axvline(0.5, color="k", ls="--", lw=1)
ax.set_xlabel("Correlación de Pearson (CIBERSORT vs DeconRNASeq)")
ax.set_title("Concordancia entre métodos por tipo celular")
plt.tight_layout()
plt.show()

print(f"Correlación media: {corrs.mean():.3f}")
print("Tipos con peor concordancia:", list(corrs.head(3).index))

## 4. Métodos basados en rangos: ssGSEA y SingScore

Estos métodos no resuelven una regresión: **rankean los genes** de cada muestra y miden si los genes característicos de cada tipo celular están en los extremos del ranking (como el ssGSEA de la práctica 5). Devuelven **scores** (no fracciones), útiles para comparar entre pacientes pero no para estimar proporciones absolutas.

El gene set por defecto (`td.read_geneset()`) deriva de los perfiles single-cell de LM22 (Le et al., 2020).

In [ ]:
# Gene sets up-regulados por tipo celular
up_geneset = td.read_geneset()
print(f"{len(up_geneset)} tipos celulares")
for ct in list(up_geneset)[:3]:
    print(f"  {ct}: {len(up_geneset[ct])} genes")

In [ ]:
# ssGSEA sobre TODOS los pacientes (~1-2 min)
ssgsea_scores = td.tumor_deconvolve(
    rna, "ssGSEA",
    patient_IDs="ALL",
    up_genes=up_geneset,
    args={"alpha": 0.5, "norm": True, "print_progress": False},
)
print("ssGSEA:", ssgsea_scores.shape)
ssgsea_scores.to_csv("ssgsea_results.csv", index_label="Patient_ID")
ssgsea_scores.iloc[:5, :8]

In [ ]:
# SingScore sobre TODOS los pacientes
singscore_scores = td.tumor_deconvolve(
    rna, "singscore",
    patient_IDs="ALL",
    up_genes=up_geneset,
)
print("SingScore:", singscore_scores.shape)
singscore_scores.to_csv("singscore_results.csv", index_label="Patient_ID")
singscore_scores.iloc[:5, :8]

In [ ]:
# Concordancia ssGSEA vs SingScore por tipo celular
corrs_rank = {}
for ct in ssgsea_scores.columns:
    r, _ = st.pearsonr(ssgsea_scores[ct], singscore_scores[ct])
    corrs_rank[ct] = r
corrs_rank = pd.Series(corrs_rank).sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(corrs_rank.index, corrs_rank.values, color="seagreen", edgecolor="k")
ax.set_xlabel("Correlación de Pearson (ssGSEA vs SingScore)")
ax.set_title("Concordancia entre métodos de rangos")
plt.tight_layout()
plt.show()
print(f"Correlación media: {corrs_rank.mean():.3f}")

## 5. Visualización de resultados

TumorDecon incluye funciones de visualización que **agrupan tipos celulares relacionados** (p. ej. suma los 3 macrófagos M0/M1/M2 en "Macrophages"). Trabajamos con las fracciones de CIBERSORT.

In [ ]:
# Boxplot: distribución de cada tipo celular en la cohorte
td.cell_frequency_boxplot(
    ciber_freqs,
    title="Infiltración inmune en TCGA COADREAD (CIBERSORT)",
    font_scale=0.75,
    title_fontsize=13,
    save_as="boxplot_cibersort.png",
)
plt.show()

In [ ]:
# Barchart: composición de pacientes individuales (subset de 12)
pacientes_ejemplo = list(ciber_freqs.index[:12])
td.cell_frequency_barchart(
    ciber_freqs.loc[pacientes_ejemplo],
    title="Composición inmune por paciente",
    save_as="barchart_cibersort.png",
)
plt.show()

In [ ]:
# Pair plot: correlaciones entre tipos celulares (subset para que sea legible)
td.pair_plot(ciber_freqs.loc[pacientes_ejemplo], save_as="pairplot_cibersort.png")
plt.show()

In [ ]:
# Clustermap: ¿se agrupan pacientes con perfiles inmunes similares?
td.hierarchical_clustering(
    ciber_freqs.iloc[:60],
    title="Clustering de pacientes por perfil inmune",
    figsize=(12, 12),
    save_as="clustermap_cibersort.png",
)
plt.show()

In [ ]:
# Agrupar tipos celulares relacionados manualmente
dict_macro = {"Macrophages": ["Macrophages M0", "Macrophages M1", "Macrophages M2"]}
print("Ejemplo: unir los 3 tipos de macrófagos")
print(td.combine_celltypes(ciber_freqs.iloc[:5], cols_to_combine=dict_macro).iloc[:, -3:])

# Simplificación automática (categorías por defecto de LM22)
ciber_simple = td.combine_celltypes(ciber_freqs)
print(f"\nDe {ciber_freqs.shape[1]} a {ciber_simple.shape[1]} columnas:")
print(list(ciber_simple.columns))

## 6. Matriz de signaturas alternativa (opcional)

Además de LM22, podemos usar cualquier matriz de signaturas propia. `kmeans_signature_matrix_qval.txt` contiene 16 subtipos celulares derivados de datos single-cell (curso de la asignatura). Está en **IDs Ensembl**, que `read_sig_file` convierte a símbolos Hugo automáticamente.

In [ ]:
# Cargar la matriz alternativa (conversión Ensembl -> Hugo automática)
sig2 = td.read_sig_file(DATA_DIR / "kmeans_signature_matrix_qval.txt", geneID="Ensembl_Gene_ID")
print(f"Matriz alternativa: {sig2.shape[0]} genes x {sig2.shape[1]} subtipos")
print(list(sig2.columns))

comunes2 = sig2.index.intersection(rna.index)
print(f"\nGenes de la matriz alternativa presentes en la mezcla: {len(comunes2)} de {sig2.shape[0]}")

In [ ]:
# CIBERSORT con la matriz alternativa (subset de 12 pacientes)
ciber_sig2 = td.tumor_deconvolve(
    rna, "cibersort",
    patient_IDs=pacientes_ejemplo,
    sig_matrix=sig2,
    args={"nu": "best", "scaling": "minmax"},
)
ciber_sig2

## 7. Exportación de resultados

In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile("resultados_deconvolucion.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["cibersort_results.csv", "deconrnaseq_results.csv",
              "ssgsea_results.csv", "singscore_results.csv",
              "boxplot_cibersort.png", "barchart_cibersort.png",
              "pairplot_cibersort.png", "clustermap_cibersort.png"]:
        try:
            z.write(f)
        except FileNotFoundError:
            print(f"(no existe {f}, se omite)")
print("ZIP listo")
files.download("resultados_deconvolucion.zip")

## 8. Preguntas de reflexión

1. **Tipos dominantes**: según CIBERSORT, ¿qué tipos celulares inmunes dominan el microambiente del cáncer colorrectal? ¿Qué fracción media tienen los macrófagos?
2. **Concordancia de métodos**: ¿qué tipos celulares tienen peor concordancia entre CIBERSORT y DeconRNASeq? ¿Por qué crees que son difíciles de distinguir (pista: mirad cuántos genes de LM22 los separan)?
3. **Fracciones vs scores**: ¿por qué los métodos de rangos no pueden dar proporciones absolutas? ¿Qué ventaja tienen frente a los modelos lineales?
4. **M1 vs M2**: los macrófagos M1 (proinflamatorios, antitumorales) y M2 (inmunosupresores, protumorales) tienen signaturas parecidas. ¿Confías en la distinción M1/M2 de CIBERSORT? ¿Cómo la validarías?
5. **Clustermap**: ¿aparecen grupos de pacientes con perfiles de infiltración distintos ("calientes" vs "fríos")? ¿Qué implicación clínica podría tener?
6. **Matriz alternativa**: ¿cómo cambian los resultados con la matriz de 16 subtipos? ¿Qué limitación tiene usar menos genes compartidos?

---

**Entrega**: el notebook ejecutado (o script `.py`) + un PDF con las figuras clave y las respuestas a las preguntas.